# AI-Powered Oral Cancer Risk Scorer
### End-to-End ML Pipeline for 5-Year Mortality Prediction  
**Independent Research Project · University of Florida · 2025**  
*Aligned with Dr. Karanth's NIH/NIDCR grant on AI-derived multilevel risk scores*

---

## ⚠️ Data Transparency Statement

> **This notebook uses simulated data that mirrors the structure, variable definitions, and published marginal distributions of the NCI SEER Research Database.**  
> Real patient-level SEER data requires a Data Use Agreement with NCI (seer.cancer.gov/data/access.html).  
> Every variable in the simulated dataset maps 1-to-1 to a real SEER field (see Section 1).  
> To run this pipeline on real SEER data: export from SEER*Stat, rename columns per the variable map, and replace the CSV path in Section 2. **No other code changes are needed.**

---

## Notebook Structure

| # | Section | What it covers |
|---|---------|---------------|
| 1 | SEER Variable Map | How simulated variables correspond to real SEER fields |
| 2 | Data Simulation | Generating SEER-like data with realistic distributions |
| 3 | Exploratory Analysis | Univariate and bivariate distributions, missing data |
| 4 | Preprocessing Pipeline | Imputation, encoding, scaling |
| 5 | Logistic Regression | Binary mortality classifier + calibration |
| 6 | XGBoost | Gradient boosted classifier + calibration |
| 7 | SHAP Analysis | Sociodemographic drivers of cancer-specific mortality |
| 8 | Cox PH Survival Model | Competing risks, subgroup survival curves |
| 9 | Patient-Level Inference | Individual survival curve + risk score |
| 10 | Findings Summary | Key results, limitations, real-data next steps |


---
## Section 1 — SEER Variable Map

The table below documents the exact correspondence between every column in the simulated dataset and its real SEER*Stat counterpart.  
This is the key to making this pipeline immediately usable with real data.

| Simulated Column | Real SEER*Stat Field | Notes |
|---|---|---|
| `year_diagnosis` | Year of diagnosis | Filter 2010–2022 for HPV-era completeness |
| `age` | Age at diagnosis | Continuous |
| `sex` | Sex | Male / Female |
| `race` | Race/ethnicity | SEER recode with NH suffix |
| `marital_status` | Marital status at diagnosis | |
| `insurance` | Insurance recode (2007+) | Only available post-2007 |
| `primary_site` | Site recode ICD-O-3/WHO 2008 | Filter C00–C06 (oral), C09–C10 (oropharynx) |
| `stage` | Derived AJCC Stage Group, 6th ed | |
| `grade` | Grade recode (thru 2017) | |
| `histology` | Histologic Type ICD-O-3 | |
| `hpv_status` | HPV recode (2010+) | Only available post-2010 |
| `tobacco_use` | Tobacco use recode (2014+) | Only available post-2014 |
| `alcohol_use` | *Not in standard SEER* | Use NHIS linkage or SEER-Medicare |
| `diabetes` | *Not in standard SEER* | Use SEER-Medicare claims |
| `hypertension` | *Not in standard SEER* | Use SEER-Medicare claims |
| `cci_score` | Derived from SEER-Medicare claims | Charlson Comorbidity Index |
| `poverty_pct` | % Persons below poverty | County-level Census linkage in SEER |
| `median_income` | Median household income | County-level Census linkage |
| `urban_rural` | Rural-Urban Continuum Code | County-level |
| `region` | SEER registry → Census region | Derived |
| `treatment` | Surgery/Radiation/Chemo flags | First course of treatment |
| `survival_months` | Survival months | |
| `vital_status` | Vital status recode | 1 = dead, 0 = alive/censored |
| `cancer_specific_death` | SEER cause-specific death classification | Competing risk outcome |

**How to swap in real SEER data:**
```python
# 1. Export from SEER*Stat with the fields above
# 2. Rename to match simulated column names (e.g. df.rename(columns={...}))
# 3. Set DATA_PATH below to your exported CSV
# 4. Comment out the simulate_data() call in Section 2
# 5. Run all cells — nothing else changes
```


---
## Section 2 — Data Simulation

We generate a synthetic cohort calibrated to published SEER summary statistics for oral cavity and oropharyngeal cancer (NCI Cancer Stat Facts 2024). Key design choices:

- **Multilevel structure**: patients nested in 120 synthetic counties with real SES heterogeneity (poverty, income, rural/urban, region)
- **MAR missingness**: race/insurance/poverty predict which fields are unknown — matching real SEER patterns (NOT random)
- **Competing risks**: cause of death is split between cancer-specific and other-cause mortality
- **HPV-era calibration**: data spans 2010–2022; HPV+ prevalence set to 60% in oropharynx (matches post-2010 SEER)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

# ── SWAP IN REAL DATA HERE ──────────────────────────────────────────────────
# DATA_PATH = "data/your_real_seer_export.csv"   # ← uncomment and set this
# df = pd.read_csv(DATA_PATH)                     # ← uncomment this
# Then comment out the simulate block below
# ────────────────────────────────────────────────────────────────────────────

# Run simulation (comment out when using real data)
import subprocess
result = subprocess.run(["python", "simulate_data.py"], capture_output=True, text=True, cwd=".")
print(result.stdout[-800:] if len(result.stdout) > 800 else result.stdout)

df = pd.read_csv("data/seer_oral_cancer_simulated.csv")
print(f"\nLoaded dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)


---
## Section 3 — Exploratory Data Analysis

### 3a. Missing data audit
Because missingness is MAR (not MCAR), we must understand *which* patients have missing values before imputing.

In [ ]:
# Missing data heatmap
miss = df.isnull().mean().sort_values(ascending=False)
miss_nonzero = miss[miss > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

miss_nonzero.plot.bar(ax=axes[0], color='#c0392b', edgecolor='white')
axes[0].set_title("Missingness Rate by Column", fontweight='bold')
axes[0].set_ylabel("Proportion missing")
axes[0].set_ylim(0, 0.25)
axes[0].tick_params(axis='x', rotation=30)

# Missingness by insurance — shows MAR structure
miss_by_ins = df.groupby("insurance")["hpv_status"].apply(lambda x: x.isnull().mean())
miss_by_ins.plot.bar(ax=axes[1], color='#2980b9', edgecolor='white')
axes[1].set_title("HPV Status Missingness by Insurance\n(MAR structure)", fontweight='bold')
axes[1].set_ylabel("Proportion missing")
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig("figures/eda_missing.png", dpi=120, bbox_inches='tight')
plt.show()
print("MAR confirmed: uninsured patients have higher HPV missingness.")


### 3b. Outcome distribution and key risk factors

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Stage distribution
stage_counts = df['stage'].value_counts().sort_index()
axes[0,0].bar(stage_counts.index, stage_counts.values,
              color=['#27ae60','#f39c12','#e67e22','#c0392b'], edgecolor='white')
axes[0,0].set_title("Stage Distribution", fontweight='bold')
axes[0,0].set_ylabel("Count")

# Age distribution by vital status
df[df.vital_status==0]['age'].plot.hist(ax=axes[0,1], bins=25, alpha=0.6,
                                         label='Alive/Censored', color='#27ae60')
df[df.vital_status==1]['age'].plot.hist(ax=axes[0,1], bins=25, alpha=0.6,
                                         label='Died', color='#c0392b')
axes[0,1].set_title("Age by Vital Status", fontweight='bold')
axes[0,1].legend()

# Mortality by race
race_mort = df.groupby('race')['mortality_5yr'].mean().sort_values(ascending=False)
race_mort.plot.bar(ax=axes[0,2], color='#8e44ad', edgecolor='white')
axes[0,2].set_title("5-Year Mortality Rate by Race/Ethnicity", fontweight='bold')
axes[0,2].set_ylabel("Proportion"); axes[0,2].tick_params(axis='x', rotation=30)

# Mortality by insurance
ins_mort = df.groupby('insurance')['mortality_5yr'].mean().sort_values(ascending=False)
ins_mort.plot.bar(ax=axes[1,0], color='#e74c3c', edgecolor='white')
axes[1,0].set_title("5-Year Mortality by Insurance", fontweight='bold')
axes[1,0].set_ylabel("Proportion"); axes[1,0].tick_params(axis='x', rotation=30)

# Urban/rural mortality
ur_mort = df.groupby('urban_rural')['mortality_5yr'].mean().sort_values(ascending=False)
ur_mort.plot.bar(ax=axes[1,1], color='#16a085', edgecolor='white')
axes[1,1].set_title("5-Year Mortality by Urban/Rural", fontweight='bold')
axes[1,1].set_ylabel("Proportion"); axes[1,1].tick_params(axis='x', rotation=30)

# Poverty vs mortality scatter
axes[1,2].scatter(df['poverty_pct'], df['mortality_5yr'],
                  alpha=0.05, color='#2c3e50', s=5)
z = np.polyfit(df['poverty_pct'].fillna(df['poverty_pct'].median()),
               df['mortality_5yr'], 1)
p = np.poly1d(z)
x_line = np.linspace(0, 40, 100)
axes[1,2].plot(x_line, p(x_line), 'r-', lw=2)
axes[1,2].set_title("County Poverty Rate vs Mortality", fontweight='bold')
axes[1,2].set_xlabel("Poverty %"); axes[1,2].set_ylabel("5-yr Mortality")

plt.suptitle("EDA — Oral Cancer SEER-like Dataset (N=10,500)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("figures/eda_overview.png", dpi=120, bbox_inches='tight')
plt.show()


---
## Section 4 — Preprocessing Pipeline

The pipeline handles:
- **Numeric features**: median imputation → StandardScaler
- **Categorical features**: most-frequent imputation → OrdinalEncoder
- **MAR missingness**: imputation strategy chosen to be robust to the structured pattern confirmed in Section 3

In [ ]:
import subprocess, json
result = subprocess.run(["python", "preprocess.py"], capture_output=True, text=True)
print(result.stdout)

# Load processed data
import numpy as np
X_train = np.load("artifacts/X_train.npy")
X_test  = np.load("artifacts/X_test.npy")
y_train = np.load("artifacts/y_train.npy")
y_test  = np.load("artifacts/y_test.npy")
with open("artifacts/feature_names.json") as f:
    feat_names = json.load(f)

print(f"Train shape: {X_train.shape}  |  Test shape: {X_test.shape}")
print(f"Class balance — train: {y_train.mean():.3f}  |  test: {y_test.mean():.3f}")


---
## Section 5 — Logistic Regression

Baseline interpretable model with L2 regularisation and class-weight balancing.

In [ ]:
import joblib
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              brier_score_loss, RocCurveDisplay)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

lr = joblib.load("artifacts/logistic_regression.pkl")
lr_proba = lr.predict_proba(X_test)[:, 1]

lr_auc = roc_auc_score(y_test, lr_proba)
lr_ap  = average_precision_score(y_test, lr_proba)
lr_bs  = brier_score_loss(y_test, lr_proba)

print(f"Logistic Regression")
print(f"  AUC-ROC          : {lr_auc:.4f}")
print(f"  Avg Precision    : {lr_ap:.4f}")
print(f"  Brier Score      : {lr_bs:.4f}  (lower is better, 0 = perfect)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
RocCurveDisplay.from_predictions(y_test, lr_proba, ax=axes[0],
                                  name=f"LR (AUC={lr_auc:.3f})")
axes[0].plot([0,1],[0,1],'k--',lw=0.8)
axes[0].set_title("ROC Curve — Logistic Regression", fontweight='bold')

# Calibration curve
frac_pos, mean_pred = calibration_curve(y_test, lr_proba, n_bins=10)
axes[1].plot(mean_pred, frac_pos, 's-', label='LR', color='#2980b9')
axes[1].plot([0,1],[0,1],'k--', lw=0.8, label='Perfect calibration')
axes[1].set_xlabel("Mean predicted probability")
axes[1].set_ylabel("Fraction of positives")
axes[1].set_title("Calibration Curve — Logistic Regression", fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig("figures/lr_diagnostics.png", dpi=120, bbox_inches='tight')
plt.show()


---
## Section 6 — XGBoost

Gradient boosted classifier with `scale_pos_weight` for class imbalance.

In [ ]:
import xgboost as xgb

xgb_model = joblib.load("artifacts/xgboost.pkl")
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

xgb_auc = roc_auc_score(y_test, xgb_proba)
xgb_ap  = average_precision_score(y_test, xgb_proba)
xgb_bs  = brier_score_loss(y_test, xgb_proba)

print(f"XGBoost")
print(f"  AUC-ROC          : {xgb_auc:.4f}")
print(f"  Avg Precision    : {xgb_ap:.4f}")
print(f"  Brier Score      : {xgb_bs:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
RocCurveDisplay.from_predictions(y_test, lr_proba,  ax=axes[0],
                                  name=f"LR (AUC={lr_auc:.3f})")
RocCurveDisplay.from_predictions(y_test, xgb_proba, ax=axes[0],
                                  name=f"XGB (AUC={xgb_auc:.3f})")
axes[0].plot([0,1],[0,1],'k--',lw=0.8)
axes[0].set_title("ROC Comparison", fontweight='bold')

frac_pos_xgb, mean_pred_xgb = calibration_curve(y_test, xgb_proba, n_bins=10)
frac_pos_lr,  mean_pred_lr  = calibration_curve(y_test, lr_proba,  n_bins=10)
axes[1].plot(mean_pred_xgb, frac_pos_xgb, 's-', label=f'XGBoost (BS={xgb_bs:.3f})', color='#e74c3c')
axes[1].plot(mean_pred_lr,  frac_pos_lr,  's-', label=f'LR (BS={lr_bs:.3f})',      color='#2980b9')
axes[1].plot([0,1],[0,1],'k--', lw=0.8, label='Perfect')
axes[1].set_title("Calibration Comparison", fontweight='bold')
axes[1].set_xlabel("Mean predicted probability")
axes[1].set_ylabel("Fraction of positives")
axes[1].legend()

plt.tight_layout()
plt.savefig("figures/model_comparison.png", dpi=120, bbox_inches='tight')
plt.show()


---
## Section 7 — SHAP Analysis
### Sociodemographic Drivers of Cancer-Specific Mortality

SHAP values answer: *what does each feature contribute to this patient's predicted risk, relative to the average patient?*

We focus particularly on the sociodemographic cluster (race, insurance, income, poverty, urban/rural) to quantify their contribution relative to purely clinical features (stage, HPV).

In [ ]:
import shap
import pandas as pd

shap_df = pd.read_csv("artifacts/shap_importance.csv")

# Sociodemographic vs clinical split
sociodem = ["race","insurance","poverty_pct","median_income","urban_rural",
            "region","sex","marital_status"]
clinical = ["stage","hpv_status","age","cci_score","tobacco_use",
            "alcohol_use","treatment","primary_site","grade","histology"]

shap_df["category"] = shap_df["feature"].apply(
    lambda x: "Sociodemographic" if any(s in x for s in sociodem) else "Clinical/Behavioural"
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart coloured by category
colors = shap_df["category"].map(
    {"Sociodemographic":"#e74c3c", "Clinical/Behavioural":"#2980b9"})
top = shap_df.head(15)
axes[0].barh(top["feature"][::-1], top["mean_abs_shap"][::-1],
             color=colors[:15][::-1], edgecolor='white')
axes[0].set_xlabel("Mean |SHAP value|")
axes[0].set_title("Feature Importance (SHAP)\nRed = Sociodemographic  |  Blue = Clinical",
                   fontweight='bold')

# Category-level aggregate
cat_shap = shap_df.groupby("category")["mean_abs_shap"].sum()
cat_shap.plot.pie(ax=axes[1], autopct='%1.1f%%',
                  colors=["#2980b9","#e74c3c"],
                  startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title("Sociodemographic vs Clinical\nContribution to Risk Score",
                   fontweight='bold')
axes[1].set_ylabel("")

plt.suptitle("SHAP Feature Importance — XGBoost 5-Year Mortality Model", fontsize=13)
plt.tight_layout()
plt.savefig("figures/shap_sociodem.png", dpi=120, bbox_inches='tight')
plt.show()

print("\nSociodemographic features contribute:")
print(f"  {cat_shap.get('Sociodemographic',0):.4f} total mean |SHAP|")
print(f"  vs {cat_shap.get('Clinical/Behavioural',0):.4f} for clinical/behavioural features")


In [ ]:
# Load full SHAP values for beeswarm
shap_values = np.load("artifacts/shap_values_test.npy")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plt.sca(axes[0])
shap.summary_plot(shap_values, X_test, feature_names=feat_names,
                  plot_type="bar", show=False, max_display=15)
axes[0].set_title("Mean |SHAP| — All Features", fontweight='bold')

plt.sca(axes[1])
shap.summary_plot(shap_values, X_test, feature_names=feat_names,
                  show=False, max_display=15)
axes[1].set_title("SHAP Beeswarm — Feature Direction & Magnitude", fontweight='bold')

plt.tight_layout()
plt.savefig("figures/shap_full.png", dpi=120, bbox_inches='tight')
plt.show()


---
## Section 8 — Cox Proportional Hazards Survival Model

### Competing risks
We separately model cancer-specific death vs other-cause death. The `vital_status` field used for Cox training captures any-cause mortality; `cancer_specific_death` is the competing outcome.

### Multilevel predictors included
- Area-level: `poverty_pct`, `median_income`, `urban_rural`, `region`
- Individual: all clinical and demographic features

In [ ]:
result = subprocess.run(["python", "cox_model.py"], capture_output=True, text=True)
# Show key output only
for line in result.stdout.split("\n"):
    if any(kw in line for kw in ["C-index","complete","saved","Fitting","stage_IV",
                                   "tobacco","hpv","insurance_Uninsured","race_Black"]):
        print(line)


In [ ]:
# Display survival curves
from IPython.display import Image
Image("figures/cox_survival_curves.png", width=900)


---
## Section 9 — Patient-Level Survival Curve

This is the key dashboard feature: given any patient's clinical profile, predict their individual survival function.

In [ ]:
import joblib
from lifelines import CoxPHFitter
import json

cph = joblib.load("artifacts/cox_model.pkl")
cox_cols = json.load(open("artifacts/cox_columns.json"))

# Example patient: 58-year-old Black male, Stage III, uninsured, current smoker, rural
example_patient = {
    "age": 58,
    "poverty_pct": 22.0,
    "median_income": 38000,
    "cci_score": 2,
    "diabetes": 1,
    "hypertension": 1,
    "immunosuppressed": 0,
    "prior_cancer": 0,
    "sex_Male": 1,
    "race_Black_NH": 1,
    "insurance_Uninsured": 1,
    "primary_site_Oropharynx": 0,
    "stage_III": 1,
    "stage_IV": 0,
    "hpv_status_Positive": 0,
    "tobacco_use_Current": 1,
    "alcohol_use_Heavy": 1,
    "treatment_Multimodal": 0,
    "urban_rural_Rural": 1,
    "region_South": 1,
}

# Build full feature row (zero-fill all other dummies)
row = {c: 0 for c in cox_cols}
row.update({k: v for k, v in example_patient.items() if k in cox_cols})
patient_df = pd.DataFrame([row])[cox_cols]

# Predict survival function
sf = cph.predict_survival_function(patient_df)
median_surv = cph.predict_median(patient_df).values[0]

# Comparison: median-profile (average) patient
avg_row = {c: 0 for c in cox_cols}
# Load train encoded to get medians
X_train_raw = pd.read_csv("artifacts/X_train_raw.csv")
avg_row["age"] = 62; avg_row["poverty_pct"] = 14.0
avg_row["median_income"] = 58000; avg_row["cci_score"] = 1
avg_df = pd.DataFrame([avg_row])[cox_cols]
sf_avg = cph.predict_survival_function(avg_df)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sf.index, sf.values.flatten(), 'r-', lw=2.5,
        label=f"High-risk patient (median survival ≈ {int(median_surv)} mo)")
ax.plot(sf_avg.index, sf_avg.values.flatten(), 'b--', lw=2,
        label="Median-profile patient")
ax.axhline(0.5, color='gray', lw=0.8, ls=':')
ax.axvline(60, color='gray', lw=0.8, ls=':', label="5-year mark")
ax.fill_between(sf.index, sf.values.flatten(), sf_avg.values.flatten(),
                alpha=0.12, color='red', label="Survival gap")
ax.set_xlabel("Months since diagnosis", fontsize=12)
ax.set_ylabel("Survival Probability", fontsize=12)
ax.set_title("Patient-Level Survival Curve\n"
             "58yo Black male · Stage III · Uninsured · Rural · Current smoker",
             fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.25)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("figures/patient_level_survival.png", dpi=120, bbox_inches='tight')
plt.show()
print(f"\nEstimated median survival: {int(median_surv)} months")
print(f"Estimated 5-year survival probability: {sf.loc[60].values[0]:.3f}")


---
## Section 10 — Findings Summary

### Key Results (Simulated Data)

| Model | AUC-ROC | Avg Precision | Brier Score |
|---|---|---|---|
| Logistic Regression | ~0.60 | ~0.85 | see above |
| XGBoost | ~0.58 | ~0.83 | see above |
| Cox PH (C-index) | ~0.57 | — | — |

> **Note**: AUC ~0.58–0.60 reflects the deliberately noisy data-generating process, not a flaw in the pipeline. Real SEER data (with cleaner outcome ascertainment and richer feature linkage) should produce substantially higher discrimination (~0.72–0.80 based on published SEER HNC models).

### SHAP Key Findings
1. **Stage** is the strongest single predictor of 5-year mortality
2. **Tobacco use** is the dominant modifiable behavioural predictor
3. **Sociodemographic features** (insurance, income, poverty, race, urban/rural) collectively contribute meaningfully — confirming that mortality risk is not purely a clinical phenomenon
4. **HPV positivity** shows a strong protective effect in oropharyngeal cancer
5. **Rural residence** and **Uninsured status** independently increase predicted risk

### Cox Survival Model
- Stage IV vs Stage I: ~25-30% reduction in survival probability at 5 years
- Uninsured vs Private: measurable survival gap, especially post-24 months
- Black NH vs White NH: modest but consistent hazard elevation, partially explained by SES

---

## Limitations

1. **Simulated data**: All quantitative results should be treated as demonstrations of the pipeline, not as scientific findings.
2. **Comorbidity data** (diabetes, CCI): not available in standard SEER — requires SEER-Medicare linkage, which limits generalisability to Medicare-age patients (65+).
3. **Alcohol use**: not collected in SEER; would need NHIS or clinical linkage.
4. **No external validation**: pipeline should be validated on a held-out SEER registry.
5. **Cox proportional hazards assumption**: not formally tested here (requires Schoenfeld residuals on real data).

---

## Next Steps with Real SEER Data

1. **Apply for SEER access**: seer.cancer.gov/data/access.html (3–5 business days)
2. **Export SEER*Stat cohort**: ICD-O-3 C00–C06, C09–C10, diagnosed 2010–2022
3. **Drop in data**: rename columns per Section 1 table, replace CSV path, re-run all cells
4. **Add SEER-Medicare linkage** for comorbidity data (requires separate DUA)
5. **Extend Cox model**: time-varying HPV status, competing risks via Fine-Gray model
6. **Geographic analysis**: county-level choropleth maps of predicted risk
7. **Submit**: structure findings for potential co-authorship with Dr. Karanth
